In [ ]:
import pandas as pd
import random
import re
import nltk

In [ ]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [10]:
# Create a dictionary containing all news articles

news_data = {
    "Sports": [
        {"Title": "India Wins Cricket Series",
         "Article": "India defeated Australia by six wickets in the final ODI after excellent batting and bowling."},

        {"Title": "Football Team Wins Championship",
         "Article": "The football club scored two late goals to secure the league title."},

        {"Title": "Olympic Gold Medal",
         "Article": "The athlete won a gold medal after breaking the national record."},

        {"Title": "Tennis Champion",
         "Article": "The player lifted the Grand Slam trophy after an exciting final."},

        {"Title": "Kabaddi Tournament",
         "Article": "The home team dominated the national kabaddi championship."}
    ],

    "Politics": [
        {"Title": "New Tax Policy",
         "Article": "The government announced a new tax policy to improve economic growth."},

        {"Title": "Election Campaign",
         "Article": "Political leaders addressed thousands of supporters during the campaign."},

        {"Title": "Parliament Session",
         "Article": "Members debated the new education bill in parliament."},

        {"Title": "Budget Announcement",
         "Article": "The finance minister presented the annual budget."},

        {"Title": "International Summit",
         "Article": "World leaders discussed climate change and trade policies."}
    ],

    "Technology": [
        {"Title": "AI Startup Launches Product",
         "Article": "A startup introduced an AI assistant for education and healthcare."},

        {"Title": "New Smartphone Released",
         "Article": "The company launched a smartphone with advanced AI features."},

        {"Title": "Cyber Security",
         "Article": "Experts warned about increasing cyber attacks across industries."},

        {"Title": "Cloud Computing",
         "Article": "Businesses are adopting cloud technology to improve efficiency."},

        {"Title": "Software Update",
         "Article": "The latest software update improves performance and security."}
    ],

    "Business": [
        {"Title": "Stock Market Rises",
         "Article": "The stock market reached a record high after strong earnings."},

        {"Title": "Company Expansion",
         "Article": "The retail company announced expansion into international markets."},

        {"Title": "Startup Funding",
         "Article": "The startup raised millions from investors."},

        {"Title": "Bank Profit",
         "Article": "The bank reported higher quarterly profits."},

        {"Title": "Electric Vehicle Industry",
         "Article": "Automobile companies increased investments in electric vehicles."}
    ],

    "Entertainment": [
        {"Title": "Movie Breaks Records",
         "Article": "The latest movie collected huge revenue worldwide."},

        {"Title": "Music Awards",
         "Article": "Popular singers won multiple awards at the annual ceremony."},

        {"Title": "New Web Series",
         "Article": "The streaming platform released a successful web series."},

        {"Title": "Film Festival",
         "Article": "International filmmakers showcased their latest productions."},

        {"Title": "Celebrity Interview",
         "Article": "The actor discussed upcoming projects during an interview."}
    ]
}

# Convert into the required format

categories = {
    category: [(item["Title"], item["Article"]) for item in articles]
    for category, articles in news_data.items()
}

In [11]:
records = []
current_id = 1


In [12]:
for label in categories.keys():
    news_items = categories[label]

    for _ in range(200):
        headline, content = random.choice(news_items)

        item = {
            "Article_ID": current_id,
            "Title": headline,
            "Article": content,
            "Category": label
        }

        records.append(item)
        current_id += 1


In [13]:
df = pd.DataFrame.from_records(records)


In [14]:
print("Dataset Shape:", df.shape)
print(df.head())

Dataset Shape: (1000, 4)
   Article_ID                      Title  \
0           1  India Wins Cricket Series   
1           2         Olympic Gold Medal   
2           3         Kabaddi Tournament   
3           4         Kabaddi Tournament   
4           5         Kabaddi Tournament   

                                             Article Category  
0  India defeated Australia by six wickets in the...   Sports  
1  The athlete won a gold medal after breaking th...   Sports  
2  The home team dominated the national kabaddi c...   Sports  
3  The home team dominated the national kabaddi c...   Sports  
4  The home team dominated the national kabaddi c...   Sports  


In [15]:
df["Text"] = df[["Title","Article"]].agg(" ".join, axis=1)

In [16]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


In [17]:
def preprocess(text):
    text = text.lower()
    text = re.sub("[^a-zA-Z\s]", " ", text)

    filtered = []
    for token in text.split():
        if token not in stop_words:
            filtered.append(lemmatizer.lemmatize(token))

    return " ".join(filtered)


<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_3540/588412204.py:3: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub("[^a-zA-Z\s]", " ", text)


In [18]:
df["Clean_Text"] = df["Text"].apply(preprocess)

In [19]:
features = df["Clean_Text"]
labels = df["Category"]


In [20]:
from sklearn.model_selection import train_test_split

split = train_test_split(
    features,
    labels,
    stratify=labels,
    random_state=42,
    test_size=0.20
)

X_train, X_test, y_train, y_test = split

In [21]:
tfidf = TfidfVectorizer()
nb_classifier = MultinomialNB()

model = Pipeline(
    steps=[
        ("tfidf", tfidf),
        ("classifier", nb_classifier)
    ]
)


In [22]:
trained_model = model.fit(X_train, y_train)

In [23]:
predictions = trained_model.predict(X_test)

In [24]:
score = accuracy_score(y_test, predictions)
print("\nAccuracy:", score)

print("\nClassification Report\n")
print(classification_report(y_test, predictions))

cm = confusion_matrix(y_test, predictions)
print("\nConfusion Matrix\n")
print(cm)



Accuracy: 1.0

Classification Report

               precision    recall  f1-score   support

     Business       1.00      1.00      1.00        40
Entertainment       1.00      1.00      1.00        40
     Politics       1.00      1.00      1.00        40
       Sports       1.00      1.00      1.00        40
   Technology       1.00      1.00      1.00        40

     accuracy                           1.00       200
    macro avg       1.00      1.00      1.00       200
 weighted avg       1.00      1.00      1.00       200


Confusion Matrix

[[40  0  0  0  0]
 [ 0 40  0  0  0]
 [ 0  0 40  0  0]
 [ 0  0  0 40  0]
 [ 0  0  0  0 40]]
